# Fund strategy classification: shares by fund count and by repo volume

In [ ]:
import re
from pathlib import Path

import pandas as pd
import pyodbc

ROOT = Path.cwd() if (Path.cwd() / "hf_Valeri.xlsx").exists() else Path.cwd().parent
funds = pd.read_excel(ROOT / "hf_Valeri.xlsx").rename(columns={"entity_id": "lei"})
funds["lei"] = funds["lei"].str.strip().str.upper()
print(len(funds), "funds,", funds["name"].isna().sum(), "missing names")

## 1. Classify by name

In [ ]:
RULES = [
    ("Fixed income / rates RV", ["FIXED INCOME", "FIRV", "RELATIVE VALUE", " RATES", "G-10",
        "GLOBAL RATES", "INFLATION", "BOND", "TERM CREDIT", "CONVEX", "TAIL RISK", "VOLATILITY"]),
    ("Global macro", ["MACRO", "ALL WEATHER", "PURE ALPHA", "OPTIMAL PORTFOLIO", "DMO"]),
    ("Credit", ["CREDIT", "ABS ", "HIGH YIELD", "DISTRESSED"]),
    ("Equity", ["EQUITY"]),
    ("Commodity", ["COMMODITY"]),
    ("Multi-strategy platform", ["MULTI-STRATEGY", "MULTI STRATEGY", "DIVERSIFIED ALPHA"]),
]

def classify(text):
    if not isinstance(text, str):
        return "Unclassified"
    t = " " + re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9\- ]", " ", text.upper())) + " "
    return next((lab for lab, kws in RULES if any(k in t for k in kws)), "Unclassified")

funds["strategy"] = funds["name"].map(classify)
funds["strategy"].value_counts()

## 2. Total repo volume per fund (SFTDS)

In [ ]:
cnxn = pyodbc.connect('DSN=Hermes_DSN', autocommit=True)

df = pd.read_csv('C:\\Users\\hermesf\\Projects\\JobMarket\\Data\\bond_timeseries_v2.csv')
df['collateral_country'] = df['ISIN'].str[:2]
securities = tuple(df[df['collateral_country'].isin(['DE', 'IT'])]['ISIN'].unique())

In [ ]:
query = f"""

SELECT borrower_id as entity_id,
sum(nominal_euro) as volume_long

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01'
AND nominal_ccy IN ('EUR')
AND central_clearing = 'non-cleared'
AND borrower_country_residence = 'KY' AND s_borrower.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
GROUP BY borrower_id

"""

df_long = pd.read_sql_query(query, cnxn)

In [ ]:
query = f"""

SELECT lender_id as entity_id,
sum(nominal_euro) as volume_short

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01'
AND nominal_ccy IN ('EUR')
AND central_clearing = 'non-cleared'
AND lender_country_residence = 'KY' AND s_lender.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
GROUP BY lender_id

"""

df_short = pd.read_sql_query(query, cnxn)

In [ ]:
vol = df_long.merge(df_short, on='entity_id', how='outer')
vol[['volume_long', 'volume_short']] = vol[['volume_long', 'volume_short']].fillna(0)
vol['volume'] = vol['volume_long'] + vol['volume_short']
funds = funds.merge(vol[['entity_id', 'volume']], left_on='lei', right_on='entity_id', how='left')
funds['volume'] = funds['volume'].fillna(0)

## 3. Table

In [ ]:
tab = (funds.groupby('strategy')
       .agg(n_funds=('lei', 'count'), volume_bn=('volume', lambda v: v.sum() / 1e9))
       .reset_index())
tab['share_funds'] = (100 * tab['n_funds'] / tab['n_funds'].sum()).round(1)
tab['share_volume'] = (100 * tab['volume_bn'] / tab['volume_bn'].sum()).round(1)
tab = tab.sort_values('share_volume', ascending=False)
tab.to_csv(ROOT / "fund_classification_table.csv", index=False)
print(tab.to_string(index=False), "\n")
for _, r in tab.iterrows():
    print(f"{r['strategy']} & {r['n_funds']} & {r['share_funds']} & {r['share_volume']} \\\\")